# M25CSA028 - ResNet-18 Evaluation (GPU)
Steps: clone repo, load model + data from Drive, full evaluation, confusion matrix, single-image inference on `data/test/5/340.png`, push results to GitHub.

## Step 1 - Set Your GitHub Details

In [ ]:
GITHUB_USERNAME = 'your_github_username'  # e.g. johnsmith
GITHUB_REPO     = 'M25CSA028'
GITHUB_TOKEN    = 'your_personal_access_token'  # GitHub PAT (repo scope)
GIT_EMAIL       = 'your_email@example.com'
GIT_NAME        = 'M25CSA028'

## Step 2 - Mount Drive and Install Packages

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted!')

In [ ]:
!pip install -q torch torchvision numpy scikit-learn Pillow matplotlib seaborn

## Step 3 - Clone GitHub Repo

In [ ]:
import os

REPO_URL = f'https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{GITHUB_REPO}.git'
REPO_DIR = f'/content/{GITHUB_REPO}'

if os.path.exists(REPO_DIR):
    os.system(f'cd {REPO_DIR} && git pull')
else:
    os.system(f'git clone {REPO_URL} {REPO_DIR}')

os.chdir(REPO_DIR)
print('Working dir:', os.getcwd())

## Step 4 - Copy Model and Data from Google Drive
> Upload `setA.pth` and `data/` to `MyDrive/M25CSA028/` before running

In [ ]:
import shutil, os

DRIVE_BASE = '/content/drive/MyDrive/M25CSA028'

shutil.copy(f'{DRIVE_BASE}/setA.pth', 'setA.pth')
print('setA.pth copied!')

if os.path.exists('data'):
    shutil.rmtree('data')
shutil.copytree(f'{DRIVE_BASE}/data', 'data')
print('data/ copied!')

## Step 5 - Load Model

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import os
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

DATA_DIR    = 'data/test/'
MODEL_PATH  = 'setA.pth'
BATCH_SIZE  = 32
NUM_CLASSES = 10
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

dataset     = datasets.ImageFolder(DATA_DIR, transform=transform)
dataloader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=2, pin_memory=True)
class_names = dataset.classes
print('Classes:', class_names)

model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model = model.to(DEVICE)
model.eval()
print('Model loaded!')

## Step 6 - Overall Accuracy, Per-Class Accuracy and Report

In [ ]:
all_preds, all_labels = [], []

with torch.no_grad():
    for images, labels in dataloader:
        outputs = model(images.to(DEVICE))
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

overall_acc = accuracy_score(all_labels, all_preds)
macro_f1    = f1_score(all_labels, all_preds, average='macro', zero_division=0)
print(f'Overall Accuracy : {overall_acc*100:.2f}%')
print(f'Macro F1 Score   : {macro_f1:.4f}')

print('\n' + '='*50)
print('Per-Class Accuracy')
print('='*50)
for idx, name in enumerate(class_names):
    mask = all_labels == idx
    acc  = np.sum(all_preds[mask] == idx) / np.sum(mask) * 100
    tag  = ' <<< CLASS 5' if name == '5' else ''
    print(f'  Class {name:>3}: {acc:6.2f}%  (support: {np.sum(mask)}){tag}')
print('='*50)

report = classification_report(all_labels, all_preds,
                               target_names=class_names, zero_division=0)
print('\nClassification Report:\n', report)

os.makedirs('results', exist_ok=True)
with open('results/classification_report.txt', 'w') as f:
    f.write(f'Overall Accuracy : {overall_acc*100:.2f}%\n')
    f.write(f'Macro F1 Score   : {macro_f1:.4f}\n\nPer-Class Accuracy:\n')
    for idx, name in enumerate(class_names):
        mask = all_labels == idx
        acc  = np.sum(all_preds[mask] == idx) / np.sum(mask) * 100
        f.write(f'  Class {name}: {acc:.2f}%  (support: {np.sum(mask)})\n')
    f.write('\n' + report)
print('Saved: results/classification_report.txt')

## Step 7 - Confusion Matrix

In [ ]:
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix - Test Set (setA.pth)', fontsize=14)
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.tight_layout()
plt.savefig('results/confusion_matrix.png', dpi=150)
plt.show()
print('Saved: results/confusion_matrix.png')

## Step 8 - Single Image Inference: data/test/5/340.png

In [ ]:
def predict_single_image(path):
    img    = Image.open(path).convert('RGB')
    tensor = transform(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        out        = model(tensor)
        probs      = torch.softmax(out, dim=1)
        conf, pred = torch.max(probs, 1)
    print('--- Single Image Inference ---')
    print(f'Image           : {path}')
    print(f'True Class      : 5 (from folder name)')
    print(f'Predicted Class : {class_names[pred.item()]}')
    print(f'Confidence      : {conf.item()*100:.2f}%')
    print('All Class Probabilities:')
    for i, p in enumerate(probs[0].tolist()):
        marker = ' <- predicted' if i == pred.item() else ''
        print(f'  Class {class_names[i]}: {p*100:6.2f}%{marker}')
    plt.figure(figsize=(3, 3))
    plt.imshow(img)
    plt.title(f'True:5  Pred:{class_names[pred.item()]} ({conf.item()*100:.1f}%)')
    plt.axis('off')
    plt.savefig('results/single_image_result.png', dpi=100, bbox_inches='tight')
    plt.show()

predict_single_image('data/test/5/340.png')

## Step 9 - Push Results to GitHub

In [ ]:
import subprocess

subprocess.run(['git', 'config', 'user.email', GIT_EMAIL])
subprocess.run(['git', 'config', 'user.name',  GIT_NAME])
subprocess.run(['git', 'add',
    'results/confusion_matrix.png',
    'results/classification_report.txt',
    'results/single_image_result.png',
    'evaluate.py'])
subprocess.run(['git', 'commit', '-m',
    'Add GPU eval results: confusion matrix, per-class accuracy, class-5 inference'])
result = subprocess.run(['git', 'push', 'origin', 'main'],
                        capture_output=True, text=True)
print(result.stdout)
print(result.stderr)
print('Done!')